In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import numpy as np
import re

In [3]:
options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [6]:
# load product href csvs
red_df = pd.read_csv("../data/Red_wine_hrefs.csv")
white_df = pd.read_csv("../data/White_wine_hrefs.csv")
sparkling_df = pd.read_csv("../data/Champagne-Sparkling_wine_hrefs.csv")

# Combine the DataFrames into a single DataFrame
wine_df = pd.concat([red_df, white_df, sparkling_df], ignore_index=True)

In [9]:
wine_df

,href,type
0,https://www.danmurphys.com.au/product/DM_17275...,Red
1,https://www.danmurphys.com.au/product/DM_10535...,Red
2,https://www.danmurphys.com.au/product/DM_90074...,Red
3,https://www.danmurphys.com.au/product/DM_90104...,Red
4,https://www.danmurphys.com.au/product/DM_14446...,Red
...,...,...
6542,https://www.danmurphys.com.au/product/DM_13411...,Champagne-Sparkling
6543,https://www.danmurphys.com.au/product/DM_12199...,Champagne-Sparkling
6544,https://www.danmurphys.com.au/product/DM_10851...,Champagne-Sparkling
6545,https://www.danmurphys.com.au/product/DM_10750...,Champagne-Sparkling


# Going to Scrape:
### product description x
### type (of grapes), separated by space x
### vintage x
### state x
### region x
### wine sweetness
### wine body
### winery name x
### product brand x
### product name x
### non-member price x
### member price x

In [4]:
%%time

hrefs_subset= []
product_brands = []
product_names = []
colours = []
product_descriptions = []
types = []
vintages = []
states = []
regions = []
wineries = []
sizes = []
non_member_prices = []
member_prices = []
bodies = []
sweetnesses = []

total_wines = len(wine_df)
starting_i = 5052
try:
    for i in range(starting_i, len(wine_df), 1):
        
        # get href
        href = wine_df.iloc[i]["href"]
        #hrefs_subset.append(href)
        driver.get(href)
    
        #colours.append(wine_df.iloc[i]["type"])
        
        is_loaded = False
        try:
            # expand specifications list
            wait = WebDriverWait(driver, 10)
            expansion_panel = wait.until(EC.element_to_be_clickable((By.XPATH, "//mat-expansion-panel[contains(., 'Product specifications')]")))
            expansion_panel.click()
            is_loaded = True
        except TimeoutException:
            print("!!! no product page for i =", i, "--href", href)
    
        if is_loaded:    
            # Scrape product brand and name
            product_brands.append(driver.find_element(By.CLASS_NAME, "product-title__brand").get_attribute("innerText"))
            product_names.append(driver.find_element(By.CLASS_NAME, "product-title__name").get_attribute("innerText"))
        
        
            # scrape product description
            product_descriptions.append(driver.find_element(By.CLASS_NAME, "product-skeleton__details-description-reg").get_attribute("innerText"))
            
            
            # scrape specifications list
            specification_ul = driver.find_element(By.CLASS_NAME, "product-attribute__items")
            specification_li_elements = specification_ul.find_elements(By.TAG_NAME, "li")
            specifications = {}
            for li in specification_li_elements:
                key = li.find_element(By.CLASS_NAME, "product-attribute__item-key").get_attribute("innerText")
                value = li.find_element(By.CLASS_NAME, "product-attribute__item-value").get_attribute("innerText")
                specifications[key] = value
            
            if "Type" in specifications:
                types.append(specifications["Type"])
            else:
                types.append(np.nan)
            
            if "Current Vintage" in specifications:
                vintages.append(specifications["Current Vintage"])
            else:
                vintages.append(np.nan)
            
            if "State" in specifications:
                states.append(specifications["State"])
            else:
                states.append(np.nan)
            
            if "Region" in specifications:
                regions.append(specifications["Region"])
            else:
                regions.append(np.nan)
            
            if "Wine Maker" in specifications:
                wineries.append(specifications["Wine Maker"])
            else:
                wineries.append(np.nan)
            
            if "Size" in specifications:
                sizes.append(specifications["Size"])
            else:
                sizes.append(np.nan)
        
            if "Wine Sweetness" in specifications:
                sweetnesses.append(specifications["Wine Sweetness"])
            else:
                sweetnesses.append(np.nan)
        
            if "Wine Body" in specifications:
                bodies.append(specifications["Wine Body"])
            else:
                bodies.append(np.nan)
        
        
            # Scrape prices
            pattern = r'\d+\.*\d*' # regex pattern
            price_contect = driver.find_element(By.ID, "mat-tab-content-1-0")
            try:
                non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price-offer").get_attribute("innerText")
                non_member_price = re.search(pattern, non_member_price).group()
                member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
                member_price = re.search(pattern, member_price).group()
            except NoSuchElementException:
                try:
                    non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
                    non_member_price = re.search(pattern, non_member_price).group()
                    member_price = np.nan
                    print("no member price")
                except NoSuchElementException:
                    non_member_price = np.nan
                    member_price = np.nan
                    print("no price found for: i =", i, href)
            
            non_member_prices.append(non_member_price)
            member_prices.append(member_price)
        
        else:
            product_brands.append(np.nan)
            product_names.append(np.nan)
            product_descriptions.append(np.nan)
            types.append(np.nan)
            vintages.append(np.nan)
            states.append(np.nan)
            regions.append(np.nan)
            wineries.append(np.nan)
            sizes.append(np.nan)
            non_member_prices.append(np.nan)
            member_prices.append(np.nan)
            bodies.append(np.nan)
            sweetnesses.append(np.nan)  
            
        hrefs_subset.append(href)
        colours.append(wine_df.iloc[i]["type"])
        print("scraped i =", i, "of", total_wines-1)

    last_i_scraped = i

except Exception as e:
    print("Stopped scraping at i =", i)
    print(e)
    

no member price
scraped i = 5052 of 6546
no member price
scraped i = 5053 of 6546
no member price
scraped i = 5054 of 6546
no member price
scraped i = 5055 of 6546
no member price
scraped i = 5056 of 6546
no member price
scraped i = 5057 of 6546
no member price
scraped i = 5058 of 6546
no member price
scraped i = 5059 of 6546
no member price
scraped i = 5060 of 6546
no member price
scraped i = 5061 of 6546
no member price
scraped i = 5062 of 6546
no member price
scraped i = 5063 of 6546
no member price
scraped i = 5064 of 6546
no member price
scraped i = 5065 of 6546
no member price
scraped i = 5066 of 6546
no member price
scraped i = 5067 of 6546
no member price
scraped i = 5068 of 6546
no member price
scraped i = 5069 of 6546
no member price
scraped i = 5070 of 6546
no member price
scraped i = 5071 of 6546
no member price
scraped i = 5072 of 6546
no member price
scraped i = 5073 of 6546
no member price
scraped i = 5074 of 6546
no member price
scraped i = 5075 of 6546
no member price


In [5]:
cut_off = i - starting_i # make sure all lists are the same length

populated_wine_df = {
    "href" : hrefs_subset[:cut_off],
    "product_brand" : product_brands[:cut_off],
    "product_name" : product_names[:cut_off],
    "product_description" : product_descriptions[:cut_off],
    "colour" : colours[:cut_off],
    "type" : types[:cut_off],
    "vintage" : vintages[:cut_off],
    "state" : states[:cut_off],
    "region" : regions[:cut_off],
    "winery" : wineries[:cut_off],
    "size" : sizes[:cut_off],
    "non_member_price" : non_member_prices[:cut_off],
    "member_price" : member_prices[:cut_off],
    "body" : bodies[:cut_off],
    "sweetness" : sweetnesses[:cut_off]
}

populated_wine_df = pd.DataFrame(populated_wine_df)

In [6]:
populated_wine_df

,href,product_brand,product_name,product_description,colour,type,vintage,state,region,winery,size,non_member_price,member_price,body,sweetness
0,https://www.danmurphys.com.au/product/DM_ER_20...,Cyoumonkyo,Junmai Daiginjo 50 Gold 2023,The company was founded in 1913 near the sceni...,White,Rice Wine,2023,NaN,NaN,NaN,720ML,70,NaN,NaN,NaN
1,https://www.danmurphys.com.au/product/DM_ER_20...,Shoryu No Mai,Junmai Daiginjo 2023,A large typhoon hit the Ise region in 1989. Am...,White,Rice Wine,2023,NaN,NaN,NaN,720ML,330,NaN,NaN,NaN
2,https://www.danmurphys.com.au/product/DM_ER_20...,Nagomi,Junmai Daiginjo Gift box 720mL,"The raw rice used is Yamada Nishiki, which is ...",White,Rice Wine,2023,NaN,NaN,NaN,720ML,110,NaN,NaN,NaN
3,https://www.danmurphys.com.au/product/DM_ER_20...,Nagomi,Daiginjo gift box 720mL,"Yamada Nishiki, the highest grade of sake rice...",White,Rice Wine,NaN,NaN,NaN,NaN,720ML,120,NaN,NaN,NaN
4,https://www.danmurphys.com.au/product/DM_ER_20...,Dassai,39 Gift Box 2023,Brewed by Asahi Shuzo based in Yamaguchi Prefe...,White,NaN,2023,NaN,NaN,NaN,720ML,80,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1489,https://www.danmurphys.com.au/product/DM_15764...,Bollinger,Special Cuvée 375mL,Bollinger Special Cuvée is unquestionably one ...,Champagne-Sparkling,Pinot Noir,Non Vintage,NaN,Champagne,NaN,375mL,58.99,NaN,Medium Bodied,Dry
1490,https://www.danmurphys.com.au/product/DM_13411...,Nicolas Feuillatte,Champagne Special Cuvée Vintage,Only made in the very best of vintages by wine...,Champagne-Sparkling,Chardonnay - Pinot Noir - Pinot Meunier,2015,NaN,Champagne,NaN,750mL,94.99,NaN,Light Bodied,Dry
1491,https://www.danmurphys.com.au/product/DM_12199...,Laurent-Perrier,Brut Rosé,The best selling rose Champagne in the world m...,Champagne-Sparkling,Pinot Noir,Non Vintage,NaN,Champagne,NaN,750mL,174.99,NaN,NaN,Dry
1492,https://www.danmurphys.com.au/product/DM_10851...,Mumm,Grand Cordon Rosé Champagne,"Appetizingly elegant and intense, the wine del...",Champagne-Sparkling,Pinot Noir,Non Vintage,NaN,Champagne,NaN,750mL,74.95,NaN,NaN,Dry


In [7]:
populated_wine_df.iloc[0]

href                   https://www.danmurphys.com.au/product/DM_ER_20...
product_brand                                                 Cyoumonkyo
product_name                                Junmai Daiginjo 50 Gold 2023
product_description    The company was founded in 1913 near the sceni...
colour                                                             White
type                                                           Rice Wine
vintage                                                             2023
state                                                                NaN
region                                                               NaN
winery                                                               NaN
size                                                               720ML
non_member_price                                                      70
member_price                                                         NaN
body                                               

In [8]:
# save df to csv
file_path = f"data/all_wine_data_subsets/wine_dataset_i_{starting_i}-{i-1}.csv"
populated_wine_df.to_csv(file_path, index=False)

## Debugging

In [9]:
#href

In [10]:
"""options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


product_descriptions = []
types = []
vintages = []
states = []
regions = []
wineries = []
sizes = []
product_names = []
non_member_prices = []
member_prices = []
bodies = []
sweetnesses = []


# get href
driver.get(href)

is_loaded = False
try:
    # expand specifications list
    wait = WebDriverWait(driver, 10)
    expansion_panel = wait.until(EC.element_to_be_clickable((By.XPATH, "//mat-expansion-panel[contains(., 'Product specifications')]")))
    expansion_panel.click()
    is_loaded = True
except TimeoutException:
    print("!!! no product page for:", href)  

if is_loaded:
    # scrape product description
    product_descriptions.append(driver.find_element(By.CLASS_NAME, "product-skeleton__details-description-reg").get_attribute("innerText"))
    
    
    # scrape specifications list
    specification_ul = driver.find_element(By.CLASS_NAME, "product-attribute__items")
    specification_li_elements = specification_ul.find_elements(By.TAG_NAME, "li")
    specifications = {}
    for li in specification_li_elements:
        key = li.find_element(By.CLASS_NAME, "product-attribute__item-key").get_attribute("innerText")
        value = li.find_element(By.CLASS_NAME, "product-attribute__item-value").get_attribute("innerText")
        specifications[key] = value
    
    if "Type" in specifications:
        types.append(specifications["Type"])
    else:
        types.append(np.nan)
    
    if "Current Vintage" in specifications:
        vintages.append(specifications["Current Vintage"])
    else:
        vintages.append(np.nan)
    
    if "State" in specifications:
        states.append(specifications["State"])
    else:
        states.append(np.nan)
    
    if "Region" in specifications:
        regions.append(specifications["Region"])
    else:
        regions.append(np.nan)
    
    if "Wine Maker" in specifications:
        wineries.append(specifications["Wine Maker"])
    else:
        wineries.append(np.nan)
    
    if "Size" in specifications:
        sizes.append(specifications["Size"])
    else:
        sizes.append(np.nan)
    
    
    # Scrape product name
    product_names.append(driver.find_element(By.CLASS_NAME, "product-title__name").get_attribute("innerText"))
    
    
    # Scrape prices
    pattern = r'\d+\.*\d*' # regex pattern
    price_contect = driver.find_element(By.ID, "mat-tab-content-1-0")
    try:
        non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price-offer").get_attribute("innerText")
        non_member_price = re.search(pattern, non_member_price).group()
        member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
        member_price = re.search(pattern, member_price).group()
    except NoSuchElementException:
        try:
            non_member_price = price_contect.find_element(By.CLASS_NAME, "pack__price").get_attribute("innerText")
            non_member_price = re.search(pattern, non_member_price).group()
            member_price = np.nan
            print("no member price")
        except NoSuchElementException:
            non_member_price = np.nan
            member_price = np.nan
            print("no price found for:", href)
    
    non_member_prices.append(non_member_price)
    member_prices.append(member_price)   
    
    # Scrape body and sweetness values
    profile_bars = driver.find_elements(By.CLASS_NAME, "body-profile__bar")
    body_bar = profile_bars[0]
    sweetness_bar = profile_bars[1]
    body_indicators = body_bar.find_elements(By.CSS_SELECTOR, 'body-profile-indicator')
    sweetness_indicators = sweetness_bar.find_elements(By.CSS_SELECTOR, 'body-profile-indicator')
    
    def get_indicator_value(indicators):
        for index, indicator in enumerate(indicators):
            opacity = indicator.value_of_css_property('opacity')
            if opacity == '1':
                index_with_opacity_1 = index
                return index_with_opacity_1
            else:
                return np.nan
    
    body = get_indicator_value(body_indicators)
    bodies.append(body)
    sweetness = get_indicator_value(sweetness_indicators)
    sweetnesses.append(sweetness)

else:
    product_names.append(np.nan)
    product_descriptions.append(np.nan)
    types.append(np.nan)
    vintages.append(np.nan)
    states.append(np.nan)
    regions.append(np.nan)
    wineries.append(np.nan)
    sizes.append(np.nan)
    non_member_prices.append(np.nan)
    member_prices.append(np.nan)
    bodies.append(np.nan)
    sweetnesses.append(np.nan)"""

'options = Options()\n\n#options.add_argument(\'--headless\')\noptions.add_argument(\'--no-sandbox\')\noptions.add_argument(\'--disable-dev-shm-usage\')\noptions.add_argument(\'--disable-infobars\')\noptions.add_argument(\'--disable-extensions\')\noptions.add_argument(\'--disable-notifications\')\noptions.add_argument(\'--window-size=1724,968\')\nuser_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"\noptions.add_argument(f\'user-agent={user_agent}\')\ndriver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)\n\n\nproduct_descriptions = []\ntypes = []\nvintages = []\nstates = []\nregions = []\nwineries = []\nsizes = []\nproduct_names = []\nnon_member_prices = []\nmember_prices = []\nbodies = []\nsweetnesses = []\n\n\n# get href\ndriver.get(href)\n\nis_loaded = False\ntry:\n    # expand specifications list\n    wait = WebDriverWait(driver, 10)\n    expansion_panel = wait.until(